# Downstream tasks

In [ ]:
import os
import sys
from pathlib import Path

import scanpy as sc

import warnings

warnings.filterwarnings("ignore")

In [ ]:
import interscale
from interscale.config import load_config
from interscale.evaluation.downstream_classification import (
    available_embeddings,
    classify,
    feature_sets_from_spec,
    summarize,
)
from interscale.evaluation.downstream_regression import (
    attention_pairs,
    regress_attention,
    residual_flow,
    score_against_truth,
)
from interscale.evaluation.synthetic_data import make_synthetic

interscale.tl.set_full_reproducibility()

In [ ]:
BASE_DIR_PROJECT = Path.cwd().resolve().parent.parent

sys.path.insert(0, str(BASE_DIR_PROJECT))
DATA = "synth_data_0"
RESULTS_DIR = Path(f"{BASE_DIR_PROJECT}/results/{DATA}")

## Load model

In [ ]:
adata = sc.read_h5ad(f"{BASE_DIR_PROJECT}/data/{DATA}.h5ad")
cfg = load_config(Path(f"{BASE_DIR_PROJECT}/config_files/{DATA}.yaml"))

In [ ]:
interscale.model.CombinedModel._setup_anndata(
    adata=adata,
    prediction_task=cfg.dataset.prediction_task,
    layer_key=cfg.dataset.layer_key,
    sample_key_list=cfg.dataset.sample_key,
    prediction_obs=cfg.dataset.prediction_obs,
    view_registry=False,
)

model = interscale.model.CombinedModel.load(
    os.path.join(RESULTS_DIR),
    adata,
    cfg,
    local_component=True,
    global_component=True,
)

In [ ]:
adata = model.get_model_output(adata, prefix="combined")

## Classification

In [ ]:
available_embeddings(adata)

In [ ]:
res = classify(adata, "cell_type", level="node", sample_key="slide", group_key="donor", layer="log1p_norm")
summarize(res)

In [ ]:
res_cond = classify(adata, "condition", level="graph", sample_key="slide", group_key="donor", layer="log1p_norm")
summarize(res_cond)

In [ ]:
sets = feature_sets_from_spec(
    adata,
    [
        "combined_local_emb",
        "combined_global_emb",
        "combined_local_emb+combined_global_emb",
        "X_pca",
        "expression",
    ],
)

summarize(classify(adata, "niche", feature_sets=sets, level="node", layer="log1p_norm"))

## Regression

In [ ]:
pairs = attention_pairs(
    adata,
    prefix="combined",
    sample_key="slide",
    obs_features=("cell_type", "niche", "total_counts", "n_genes_by_counts"),
    normalize="graph_z",
    max_pairs=200_000,
)

pairs.head()

In [ ]:
result = regress_attention(pairs, estimator="ridge", group_key="graph", cv_folds=5)
result.variance

In [ ]:
result.nonlinear_r2

In [ ]:
residual_flow(result.pairs)

In [ ]:
score_against_truth(adata, result.pairs)

## Create the synthetic dataset

In [ ]:
adata_synth = make_synthetic()
adata_synth

In [ ]:
adata_synth.write_h5ad(f"{BASE_DIR_PROJECT}/data/synth_data_0.h5ad")